In [ ]:
import json
import os

os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "1800")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("_CHECK_PEFT", "0")

HF_TOKEN_PATH = "/kaggle/input/imggenhub-hf-token/hf_token.json"
with open(HF_TOKEN_PATH, "r", encoding="utf-8") as hf_file:
    HF_TOKEN = json.load(hf_file)["HF_TOKEN"]

os.environ["HF_TOKEN"] = HF_TOKEN





In [ ]:
import gc
import importlib
import inspect
import os
import subprocess
import sys
from datetime import datetime

import torch
from tqdm import tqdm

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Ensure the Kaggle dataset is attached and synced.")

# -------- SETTINGS ---------
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# -------- PARAMETERS (can be overridden by deploy.py) ---------
PROMPTS = ['Fresh test: modern diffusion']
NEGATIVE_PROMPT = "blurry, distorted"
OUTPUT_DIR = "."
IMG_SIZE = (512, 512)
GUIDANCE = 4.5
PRECISION = "fp16"
STEPS = 8
SEED = 42
USE_GPU = True
REFINER_MODEL_ID = None
REFINER_STEPS = 0
REFINER_GUIDANCE = 0.0
REFINER_PRECISION = "fp16"
# ----------------------------


def detect_model_family(model_id: str) -> str:
    model_lower = (model_id or "").strip().lower()
    if any(token in model_lower for token in ("stable-diffusion-3.5", "stable-diffusion-3-5", "sd3.5", "sd3_5", "sd35")):
        return "sd35"
    if any(token in model_lower for token in ("wan2.1", "wan-2.1", "wan2_1", "wan-2", "wan2", "chroma")):
        return "wan21_chroma"
    if any(token in model_lower for token in ("qwen-image", "qwen image", "qwen-vl", "qwen2-vl", "qwen2_5-vl", "qwen")):
        return "qwen_image"
    return "stable_diffusion"


def resolve_dtype(precision: str):
    precision_lower = (precision or "").strip().lower()
    if precision_lower == "fp32":
        return torch.float32
    if precision_lower == "bf16":
        return torch.bfloat16
    return torch.float16


def get_vram_gb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 ** 3
    return 0.0


def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_pipe_call(pipe, **kwargs):
    signature = inspect.signature(pipe.__call__)
    supported = set(signature.parameters.keys())
    filtered_kwargs = {k: v for k, v in kwargs.items() if k in supported and v is not None}
    return pipe(**filtered_kwargs)


def purge_modules(prefix: str) -> None:
    stale_modules = [name for name in list(sys.modules.keys()) if name == prefix or name.startswith(f"{prefix}.")]
    for module_name in stale_modules:
        sys.modules.pop(module_name, None)


def import_diffusers_components(force_qwen_runtime: bool = False):
    if force_qwen_runtime:
        print("Installing/updating runtime dependencies for Qwen image pipelines...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "--no-deps",
            "diffusers==0.36.0",
            "transformers==4.57.1",
            "tokenizers==0.22.1",
            "huggingface-hub==0.35.3",
            "bitsandbytes==0.45.5",
        ])
        purge_modules("diffusers")
        purge_modules("transformers")
        purge_modules("accelerate")
        importlib.invalidate_caches()

    diffusers_module = importlib.import_module("diffusers")
    transformers_module = importlib.import_module("transformers")

    auto_pipeline = getattr(diffusers_module, "AutoPipelineForText2Image", None)
    diffusion_pipeline = getattr(diffusers_module, "DiffusionPipeline", None)
    sd35_pipeline = getattr(diffusers_module, "StableDiffusion3Pipeline", None)
    qwen_pipeline = getattr(diffusers_module, "QwenImagePipeline", None)

    if force_qwen_runtime:
        try:
            model_loading_utils = importlib.import_module("diffusers.models.model_loading_utils")
            if hasattr(model_loading_utils, "_caching_allocator_warmup"):
                model_loading_utils._caching_allocator_warmup = lambda *args, **kwargs: None
                print("Patched diffusers caching allocator warmup for low-memory Qwen loading.")
        except Exception as warmup_patch_error:
            print(f"Warmup patch skipped: {warmup_patch_error}")

    if qwen_pipeline is None:
        try:
            qwen_module = importlib.import_module("diffusers.pipelines.qwenimage.pipeline_qwenimage")
            qwen_pipeline = getattr(qwen_module, "QwenImagePipeline", None)
        except Exception as qwen_import_error:
            print(f"Direct qwenimage module import failed: {qwen_import_error}")

    print(f"diffusers version: {getattr(diffusers_module, '__version__', 'unknown')}")
    print(f"transformers version: {getattr(transformers_module, '__version__', 'unknown')}")
    print(f"QwenImagePipeline available: {qwen_pipeline is not None}")

    return auto_pipeline, diffusion_pipeline, sd35_pipeline, qwen_pipeline


def load_pipeline(model_id: str, model_family: str, torch_dtype, device: str):
    force_qwen_runtime = model_family == "qwen_image"
    auto_pipeline, diffusion_pipeline, sd35_pipeline, qwen_pipeline = import_diffusers_components(
        force_qwen_runtime=force_qwen_runtime
    )

    common_kwargs = {
        "torch_dtype": torch_dtype,
        "token": HF_TOKEN,
        "use_safetensors": True,
    }
    if model_family == "wan21_chroma":
        common_kwargs["trust_remote_code"] = True

    if model_family == "qwen_image":
        common_kwargs["low_cpu_mem_usage"] = True
        os.makedirs("/kaggle/working/offload", exist_ok=True)
        common_kwargs["device_map"] = "balanced"
        common_kwargs["max_memory"] = {0: "13GiB", "cpu": "48GiB"}
        common_kwargs["offload_folder"] = "/kaggle/working/offload"
        common_kwargs["offload_state_dict"] = True
        if "4bit" in (model_id or "").lower():
            common_kwargs["device_map"] = "cuda"
            common_kwargs.pop("max_memory", None)
            common_kwargs.pop("offload_folder", None)
            common_kwargs.pop("offload_state_dict", None)
        else:
            try:
                from diffusers import BitsAndBytesConfig, PipelineQuantizationConfig
                common_kwargs["quantization_config"] = PipelineQuantizationConfig(
                    quant_backend="bitsandbytes_8bit",
                    quant_mapping={
                        "transformer": BitsAndBytesConfig(
                            load_in_8bit=True,
                            llm_int8_enable_fp32_cpu_offload=True,
                        )
                    },
                )
                print("Enabled PipelineQuantizationConfig 8-bit loading for Qwen model.")
            except Exception as quantization_error:
                print(f"Quantization setup skipped: {quantization_error}")

    fallback_kwargs = dict(common_kwargs)
    fallback_kwargs.pop("use_safetensors", None)
    if model_family == "qwen_image":
        fallback_kwargs.pop("quantization_config", None)

    pipeline_loaders = []

    if model_family == "sd35":
        if sd35_pipeline is not None:
            pipeline_loaders.append(("StableDiffusion3Pipeline", sd35_pipeline.from_pretrained))
        else:
            print("StableDiffusion3Pipeline is unavailable; falling back to AutoPipelineForText2Image.")
        if auto_pipeline is not None:
            pipeline_loaders.append(("AutoPipelineForText2Image", auto_pipeline.from_pretrained))
    elif model_family == "qwen_image":
        if qwen_pipeline is not None:
            pipeline_loaders.append(("QwenImagePipeline", qwen_pipeline.from_pretrained))
        elif diffusion_pipeline is not None:
            pipeline_loaders.append(("DiffusionPipeline", diffusion_pipeline.from_pretrained))
        elif auto_pipeline is not None:
            pipeline_loaders.append(("AutoPipelineForText2Image", auto_pipeline.from_pretrained))
    else:
        if auto_pipeline is not None:
            pipeline_loaders.append(("AutoPipelineForText2Image", auto_pipeline.from_pretrained))

    if not pipeline_loaders:
        raise RuntimeError("No compatible diffusers pipeline class found in runtime.")

    last_error = None
    for pipeline_name, loader in pipeline_loaders:
        for kwargs in (common_kwargs, fallback_kwargs):
            try:
                pipe = loader(model_id, **kwargs)
                if model_family == "qwen_image" and getattr(pipe, "hf_device_map", None):
                    return pipe
                if device == "cuda":
                    if hasattr(pipe, "enable_attention_slicing"):
                        pipe.enable_attention_slicing()
                    if hasattr(pipe, "enable_vae_slicing"):
                        pipe.enable_vae_slicing()
                    if model_family in {"sd35", "wan21_chroma", "qwen_image"} and hasattr(pipe, "enable_model_cpu_offload"):
                        try:
                            pipe.enable_model_cpu_offload()
                            return pipe
                        except Exception as offload_error:
                            print(f"Model CPU offload could not be enabled: {offload_error}. Trying sequential offload.")
                    if model_family in {"sd35", "wan21_chroma", "qwen_image"} and hasattr(pipe, "enable_sequential_cpu_offload"):
                        try:
                            pipe.enable_sequential_cpu_offload()
                            return pipe
                        except Exception as offload_error:
                            print(f"Sequential CPU offload could not be enabled: {offload_error}. Falling back to direct CUDA placement.")
                    return pipe.to(device)
                return pipe.to(device)
            except Exception as exc:
                last_error = exc
                print(f"Load attempt failed with {pipeline_name}: {exc}")
                if model_family == "qwen_image" and "out of memory" in str(exc).lower():
                    cleanup_memory()
                    raise RuntimeError(f"Qwen model exceeds available memory in this runtime: {exc}")

    raise RuntimeError(f"Failed to load pipeline for model '{model_id}': {last_error}")


model_family = detect_model_family(MODEL_ID)
device = "cuda" if USE_GPU and torch.cuda.is_available() else "cpu"
dtype = resolve_dtype(PRECISION)
generator = torch.manual_seed(SEED)

print(f"Model family: {model_family}")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Prompts: {len(PROMPTS)}")
print(f"Steps: {STEPS}")
print(f"Guidance: {GUIDANCE}")
print(f"Image size: {IMG_SIZE}")
print(f"Precision: {PRECISION}")
print(f"VRAM at start: {get_vram_gb():.2f} GB")

os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, prompt in enumerate(tqdm(PROMPTS, desc="Generating")):
    print(f"\\n{'=' * 60}")
    print(f"Prompt {i + 1}/{len(PROMPTS)}: {prompt[:80]}")
    print(f"{'=' * 60}")
    print("Loading pipeline...")
    print(f"VRAM before load: {get_vram_gb():.2f} GB")

    pipe = load_pipeline(MODEL_ID, model_family, dtype, device)
    print(f"VRAM after load: {get_vram_gb():.2f} GB")

    result = safe_pipe_call(
        pipe,
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        generator=generator,
        height=IMG_SIZE[0],
        width=IMG_SIZE[1],
    )

    image = result.images[0]
    del pipe
    cleanup_memory()
    print(f"VRAM after cleanup: {get_vram_gb():.2f} GB")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"generated_{i + 1}_{timestamp}.png"
    output_path = os.path.join(OUTPUT_DIR, filename)
    image.save(output_path)
    print(f"Saved: {output_path}")

print(f"\\n{'=' * 60}")
print(f"COMPLETE! {len(PROMPTS)} images saved to {OUTPUT_DIR}")
print(f"Final VRAM: {get_vram_gb():.2f} GB")
print(f"{'=' * 60}")













